# **Speculative Decoding**

#### Paper name: **Implementation of Fast Inference from Transformers via Speculative Decoding**

In [1]:
import torch
import transformers
from transformers import (AutoTokenizer, AutoModelForCausalLM)


import time
import numpy as np
from tqdm.auto import tqdm

In [2]:
gpu = "cuda"
dtype = torch.float16
target_model = "HuggingFaceTB/SmolLM2-1.7B"
draft_model = "HuggingFaceTB/SmolLM2-360M"
max_new_tokens = 100
k = 4

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0).total_memory / 1e9)

True
NVIDIA GeForce RTX 3060 Laptop GPU
6.088359936


In [4]:
## draft and target models share the same tokenizer family, only one tokenizer is needed.

tokenizer = AutoTokenizer.from_pretrained(target_model)

print(tokenizer.vocab_size)
print(tokenizer.eos_token)
print(tokenizer.pad_token)

49152
<|endoftext|>
None


In [5]:
## tokenization demo

text = "The capital of France is"
ids = tokenizer(text).to(gpu)
print(ids)

{'input_ids': [504, 3575, 282, 4649, 314], 'attention_mask': [1, 1, 1, 1, 1]}


In [6]:
## decode back

decoded = tokenizer.decode(ids["input_ids"])
print(decoded)

The capital of France is


In [7]:
## load draft model

draft = AutoModelForCausalLM.from_pretrained(draft_model, torch_dtype=dtype).to(gpu).eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
## load target model

target = AutoModelForCausalLM.from_pretrained(target_model, torch_dtype=dtype).to(gpu).eval()

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [9]:
## Count parameters - this immediately shows why speculative decoding can be beneficial: the draft model is much
## cheaper to run than the target model, so we can use it to generate a draft and only run the target model on a subset of tokens.

draft_params = sum(p.numel() for p in draft.parameters())
target_params = sum(p.numel() for p in target.parameters())
print(draft_params, target_params, draft_params / target_params)

361821120 1711376384 0.211421124764101


In [10]:
## model configuration
"""
Inspect:
- hidden size
- attention heads
- layers
- max context
- vocab size
"""

print(draft.config)
print(target.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "float16",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 960,
  "initializer_range": 0.02,
  "intermediate_size": 2560,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 15,
  "num_hidden_layers": 32,
  "num_key_value_heads": 5,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "use_cache": true,
  "vocab_size": 49152
}

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "float16",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden

In [11]:
## forward pass
inputs = tokenizer("The capital of France is", return_tensors = "pt")
print(inputs.items())
inputs = {k: v.to(gpu) for k, v in inputs.items()}
with torch.inference_mode():
    output = target(**inputs)

print(output.keys())

ItemsView({'input_ids': tensor([[ 504, 3575,  282, 4649,  314]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])})
odict_keys(['logits', 'past_key_values'])


In [15]:
## predict next token

next_token = torch.argmax(output.logits[:, -1], dim=-1)
print(next_token)
print(tokenizer.decode(next_token))

tensor([7042], device='cuda:0')
 Paris
